In [10]:
import pandas as pd 
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('processados\dados_climaticos_tratados_mensal.csv', sep=';', encoding='latin-1')
df_doencas = pd.read_csv('resultado_final_doencas.csv', sep=";", encoding="latin1")

display(df)


<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
C:\Users\eduar\AppData\Local\Temp\ipykernel_30524\553044409.py:7: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv('processados\dados_climaticos_tratados_mensal.csv', sep=';', encoding='latin-1')


,municipio,ano,mes,precipitacao_sum,temp_orvalho_median
0,BAGE,2015,1,207.4,22.300
1,BAGE,2015,2,87.8,21.675
2,BAGE,2015,3,32.4,21.300
3,BAGE,2015,4,21.2,18.175
4,BAGE,2015,5,126.2,14.750
...,...,...,...,...,...
3514,VACARIA,2025,8,113.8,8.950
3515,VACARIA,2025,9,223.6,10.175
3516,VACARIA,2025,10,138.4,11.750
3517,VACARIA,2025,11,124.8,12.875


In [11]:
municipios_analisados = ['PORTO ALEGRE', 'SANTA MARIA', 'SANTO AUGUSTO',
                          'URUGUAIANA', 'CACAPAVA DO SUL', 'BAGE',
                          'SOLEDADE', 'LAGOA VERMELHA', 'PASSO FUNDO']

In [12]:
df_merge= df.merge(df_doencas, on=['municipio', 'mes', 'ano'], how='inner')

df_final = df_merge.copy()
df_final['precipitacao_sum'] = df_final['precipitacao_sum'].astype(float)
df_final['temp_orvalho_median'] = df_final['temp_orvalho_median'].astype(float)


df_final_agrupada = df_final.groupby(['municipio', 'ano', 'mes']).agg({
    'precipitacao_sum': 'sum',
    'internacoes': 'sum',
    'temp_orvalho_median': 'median'
}).reset_index()


In [13]:
df_final_filtrada2 = df_final_agrupada[df_final_agrupada['municipio'].isin(municipios_analisados)]
df_final_filtrada2= df_final_filtrada2.drop(df_final_filtrada2[df_final_filtrada2['precipitacao_sum'] <= 0].index)

df_final_filtrada = df_final_filtrada2.copy()

for municipio in municipios_analisados:
    df_municipio = df_final_filtrada[df_final_filtrada['municipio'] == municipio]

    #display(df_final_filtrada)

    coef, p_valor = stats.spearmanr(df_municipio['precipitacao_sum'], df_municipio['internacoes'])
    coef2, p_valor2 = stats.spearmanr(df_municipio['temp_orvalho_median'], df_municipio['internacoes'])
    
    df_corr = df_municipio[['precipitacao_sum', 'internacoes', 'temp_orvalho_median']].corr(method='spearman')
    
    df_corr['p_valor'] = None

    df_corr.loc['precipitacao_sum', 'p_valor'] = p_valor
    df_corr.loc['temp_orvalho_median', 'p_valor'] = p_valor2

    
    df_corr.to_csv(f"processados/novos/correlacao_{municipio}.csv", index=True, sep=";", encoding="latin1", decimal=".")
    

    print(f"Prep: Coeficiente: {coef:.3f} | p-valor: {p_valor:.4f}")
    print(f"Temp: Coeficiente: {coef2:.3f} | p-valor: {p_valor2:.4f}")
    print(df_corr)


Prep: Coeficiente: 0.292 | p-valor: 0.0007
Temp: Coeficiente: 0.356 | p-valor: 0.0000
                     precipitacao_sum  internacoes  temp_orvalho_median  \
precipitacao_sum             1.000000     0.291855            -0.110220   
internacoes                  0.291855     1.000000             0.355815   
temp_orvalho_median         -0.110220     0.355815             1.000000   

                      p_valor  
precipitacao_sum     0.000685  
internacoes              None  
temp_orvalho_median  0.000028  
Prep: Coeficiente: 0.144 | p-valor: 0.1024
Temp: Coeficiente: 0.439 | p-valor: 0.0000
                     precipitacao_sum  internacoes  temp_orvalho_median  \
precipitacao_sum              1.00000     0.143910             0.149820   
internacoes                   0.14391     1.000000             0.438993   
temp_orvalho_median           0.14982     0.438993             1.000000   

                      p_valor  
precipitacao_sum     0.102364  
internacoes              None  
te

C:\Users\eduar\AppData\Local\Temp\ipykernel_30524\2543446764.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  coef, p_valor = stats.spearmanr(df_municipio['precipitacao_sum'], df_municipio['internacoes'])
C:\Users\eduar\AppData\Local\Temp\ipykernel_30524\2543446764.py:12: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  coef2, p_valor2 = stats.spearmanr(df_municipio['temp_orvalho_median'], df_municipio['internacoes'])


In [14]:
df_plot_filtrado = df_final_filtrada[df_final_filtrada['internacoes'] != 0]
df_plot_filtrado = df_plot_filtrado[df_plot_filtrado['municipio'] == 'PORTO ALEGRE']
df_plot_filtrado['data'] = pd.to_datetime({
    'year': df_plot_filtrado['ano'],
    'month': df_plot_filtrado['mes'],
    'day': 1
})
df_plot_filtrado = df_plot_filtrado.groupby(['municipio', 'data']).agg({'precipitacao_sum': 'sum', 'internacoes': 'sum', 'temp_orvalho_median': 'median'}).reset_index()
display(df_plot_filtrado)

,municipio,data,precipitacao_sum,internacoes,temp_orvalho_median
0,PORTO ALEGRE,2015-01-01,159.8,5,25.350
1,PORTO ALEGRE,2015-02-01,99.4,3,24.000
2,PORTO ALEGRE,2015-03-01,54.0,9,23.450
3,PORTO ALEGRE,2015-04-01,76.0,3,20.925
4,PORTO ALEGRE,2015-05-01,142.4,9,17.900
...,...,...,...,...,...
121,PORTO ALEGRE,2025-07-01,91.2,4,10.300
122,PORTO ALEGRE,2025-08-01,211.6,4,11.400
123,PORTO ALEGRE,2025-09-01,212.0,4,13.975
124,PORTO ALEGRE,2025-11-01,39.9,5,16.200
